<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Transformation — Theory</b></h1>
</div>

## Theoretical Foundations

This notebook documents the mathematical model, estimation methods, numerical considerations, diagnostics, and limitations used in the laboratory.
### Technical Context

Intensity transformations change pixel values while geometric transformations change where image information is sampled.

### Core Transformation Model

Pointwise intensity processing follows $s=T(r)$. Geometric processing follows $\mathbf{p}'=H\mathbf{p}$ in homogeneous coordinates, with inverse mapping used to populate a discrete output grid.

### Notation and Conventions

| Symbol | Meaning |
| --- | --- |
| $r,s$ | input/output intensity |
| $I$ | image array |
| $\mathbf{p}=[x,y,1]^T$ | homogeneous image point |
| $H$ | $3\times3$ transformation matrix |
| $T$ | intensity mapping |

### Analytical Scope

Explain and implement intensity mappings, homogeneous-coordinate transforms, interpolation, transformation composition, output-canvas behavior, and validation.


## 1. Data and Output Paths

The notebook automatically locates the laboratory directory by searching upward for both `data/` and `notebooks/`.

This avoids hard-coded machine-specific paths.


## 2. Load and Inspect the Reference Images

Intensity transformations are easiest to understand on grayscale images, while geometric transformations are demonstrated on both grayscale and RGB images.


## 3. Intensity Transformation Model

An intensity transformation changes pixel **values** but does not move pixels.

For a grayscale image:

$$
s = T(r)
$$

where:

- $r$ = input intensity;
- $T$ = transformation function;
- $s$ = output intensity.

The transformation is **pointwise** when each output pixel depends only on the corresponding input pixel:

$$
J(y,x)=T(I(y,x))
$$

The spatial coordinates remain unchanged.

### Deeper understanding

A pointwise intensity transform acts independently on each pixel:

$$
s=T(r),
$$

where $r$ is the input intensity and $s$ the output intensity. For a valid monotonic contrast mapping, preserving intensity order usually requires $T'(r)\ge0$. The slope determines local contrast amplification: large slope expands nearby input levels; small slope compresses them.

Because the mapping is pointwise, image geometry is unchanged. Any spatial effect observed after an intensity transformation must therefore arise from the way intensities are redistributed, not from a coordinate change.


## 4. Image Negative

For an 8-bit grayscale image:

$$
s = 255-r
$$

Dark intensities become bright and bright intensities become dark.

The operation reverses the gray scale but preserves spatial structure.


## 5. Brightness and Contrast

A simple linear point transformation is:

$$
s = ar+b
$$

where:

- $a$ controls contrast;
- $b$ controls brightness.

Typical cases:

```text
a = 1, b > 0  → brighter
a = 1, b < 0  → darker
a > 1          → stronger contrast around zero
0 < a < 1      → compressed contrast
```

Because an 8-bit image can store only 0–255, the result must be clipped.

### Deeper understanding

A common affine intensity model is

$$
s = ar+b,
$$

where $a$ controls contrast and $b$ controls brightness. For $a>1$, intensity differences are amplified; for $0<a<1$, they are compressed. A non-zero $b$ shifts the whole distribution.

The theoretical mapping may produce values outside the representable range. Clipping,

$$
s_{\mathrm{clip}}=\min(L-1,\max(0,s)),
$$

is therefore part of the practical operator and can make the mapping irreversible when multiple input values collapse to the same boundary value.


## 6. Contrast Stretching

A low-contrast image uses only a narrow part of the available intensity range.

Min-max contrast stretching maps:

$$
r_{\min}\rightarrow 0
$$

and:

$$
r_{\max}\rightarrow 255
$$

using:

$$
s=
\frac{r-r_{\min}}
{r_{\max}-r_{\min}}
\times255
$$

This is a global linear transformation.

### Deeper understanding

For input interval $[r_{\min},r_{\max}]$ mapped to output interval $[s_{\min},s_{\max}]$,

$$
s=s_{\min}+
\frac{r-r_{\min}}{r_{\max}-r_{\min}}
(s_{\max}-s_{\min}).
$$

When robust percentiles replace the absolute minimum and maximum, isolated outliers have less influence. The method increases occupied dynamic range but does not recover information that was never recorded.


## 7. Logarithmic Transformation

The log transform is:

$$
s=c\log(1+r)
$$

The logarithm expands small input values and compresses large values.

This can reveal information stored in dark intensity regions while compressing a large dynamic range.

For an 8-bit output, $c$ is chosen so the maximum maps to 255.

### Deeper understanding

A standard logarithmic mapping is

$$
s=c\log(1+r).
$$

Its derivative,

$$
\frac{ds}{dr}=\frac{c}{1+r},
$$

is largest at low intensities and decreases as $r$ grows. The transform therefore expands dark-level differences while compressing bright-level differences. The constant $c$ is normally chosen so that the maximum mapped value fits the target dynamic range.


## 8. Gamma / Power-Law Transformation

The normalized power-law transformation is:

$$
s=r^\gamma
$$

where $r$ and $s$ are normalized to $[0,1]$.

General behavior:

- $\gamma<1$ → brightens many mid/dark values;
- $\gamma=1$ → identity;
- $\gamma>1$ → darkens many mid values.

This is closely related to gamma correction used in imaging and display systems.

### Deeper understanding

The power-law transform is

$$
s=c\,r^\gamma.
$$

For normalized intensities $r\in[0,1]$, $\gamma<1$ brightens mid-to-low intensities and $\gamma>1$ darkens them. Gamma correction is especially important because acquisition and display devices may themselves have nonlinear responses. Applying an inverse power can compensate for such device behavior when the response model is known.


## 9. Histogram Equalization

Histogram equalization is a global intensity transformation derived from the image histogram.

For an 8-bit image:

1. compute the histogram;
2. normalize it to obtain a probability distribution;
3. compute the cumulative distribution function (CDF);
4. map each input intensity through the CDF.

Conceptually:

$$
s=(L-1)\,\mathrm{CDF}(r)
$$

with $L=256$.

Unlike contrast stretching, the transform depends on the whole intensity distribution.

### Deeper understanding

For a discrete gray-level distribution $p(r_k)$, histogram equalization uses the cumulative distribution,

$$
s_k=(L-1)\sum_{j=0}^{k}p(r_j).
$$

The mapping is monotonic, so intensity ordering is preserved. The result tends toward a more widely spread histogram, but exact uniformity is not guaranteed for finite discrete images. Equalization is global: it can improve overall contrast while still failing in regions whose local contrast differs from the global distribution.


## 10. Compare the Fundamental Intensity Transformations

The following comparison emphasizes an important idea:

**every transformation changes the histogram differently while preserving pixel locations.**


## 11. Geometric Transformation Model

A geometric transformation changes **where image content is located**.

Instead of changing intensity $r\rightarrow s$, we transform coordinates:

$$
(x,y)\rightarrow(x',y')
$$

Examples include:

- translation;
- scaling;
- rotation;
- reflection;
- shear.

After the coordinate transformation, the output grid usually asks for values at locations that do not correspond exactly to integer input pixels. This is why **interpolation** becomes necessary.


## 12. Homogeneous Coordinates

A 2-D point:

$$
(x,y)
$$

is represented in homogeneous form as:

$$
\mathbf{p}
=
\begin{bmatrix}
x\\
y\\
1
\end{bmatrix}
$$

This allows translation, rotation, scaling, reflection, shear, and affine transformations to be written using matrix multiplication:

$$
\mathbf{p}'=\mathbf{H}\mathbf{p}
$$

where $\mathbf{H}$ is a $3\times3$ transformation matrix.


## 13. Fundamental Geometric Transformation Matrices

### Translation

$$
\mathbf{T}
=
\begin{bmatrix}
1&0&t_x\\
0&1&t_y\\
0&0&1
\end{bmatrix}
$$

### Scaling

$$
\mathbf{S}
=
\begin{bmatrix}
s_x&0&0\\
0&s_y&0\\
0&0&1
\end{bmatrix}
$$

### Rotation about the origin

$$
\mathbf{R}
=
\begin{bmatrix}
\cos\theta&-\sin\theta&0\\
\sin\theta&\cos\theta&0\\
0&0&1
\end{bmatrix}
$$

### Horizontal shear

$$
\mathbf{H}_x
=
\begin{bmatrix}
1&k_x&0\\
0&1&0\\
0&0&1
\end{bmatrix}
$$

### Reflection about the vertical axis

$$
\mathbf{F}_x
=
\begin{bmatrix}
-1&0&0\\
0&1&0\\
0&0&1
\end{bmatrix}
$$

The image coordinate system has $y$ increasing downward, so visual rotation direction must be interpreted carefully.

### Deeper understanding

In homogeneous coordinates, a 2-D affine transform is represented as

$$
\begin{bmatrix}
x'\\y'\\1
\end{bmatrix}
=
\begin{bmatrix}
a_{11}&a_{12}&t_x\\
a_{21}&a_{22}&t_y\\
0&0&1
\end{bmatrix}
\begin{bmatrix}
x\\y\\1
\end{bmatrix}.
$$

The upper-left $2\times2$ block controls linear geometry—rotation, anisotropic scaling, shear, and reflection—while the last column adds translation. Affine transforms preserve straight lines, parallelism, and ratios along a line, but not necessarily lengths or angles.


## 14. Origin-Centered vs Centered Geometry

Scaling and rotation matrices naturally operate around the coordinate origin `(0, 0)`.

For images, `(0, 0)` is normally the top-left corner.

To rotate or scale around the image center:

```text
1. translate the center to the origin
2. apply the transformation
3. translate back
```

In homogeneous matrices:

```text
H_centered = T(center) @ H @ T(-center)
```

### Deeper understanding

To rotate or scale about image center $c=(c_x,c_y)$ rather than the origin, use translation-conjugation:

$$
T(c)\,A\,T(-c).
$$

The rightmost transform acts first. This order moves the desired pivot to the origin, applies the linear transform, then moves coordinates back. Forgetting these translations is the most common reason a correct rotation matrix appears to “move” the image unexpectedly.


## 15. Forward Mapping vs Inverse Mapping

Suppose the forward transform is:

$$
\mathbf{p}_{out}
=
\mathbf{H}\mathbf{p}_{in}
$$

A tempting implementation is to send every input pixel to its transformed output position.

This is **forward mapping**.

The problem is that discrete transformed coordinates can leave holes in the output.

Image warping normally uses **inverse mapping**:

$$
\mathbf{p}_{in}
=
\mathbf{H}^{-1}\mathbf{p}_{out}
$$

For every output pixel:

1. ask where it came from in the input;
2. sample the input image at that location;
3. interpolate if the location is non-integer.

### Deeper understanding

Forward mapping computes $\mathbf{x}'=T\mathbf{x}$ from source pixels. Because transformed coordinates are generally non-integer and not every destination pixel receives a source sample, holes can appear.

Inverse mapping evaluates

$$
\mathbf{x}=T^{-1}\mathbf{x}'
$$

for every destination pixel $\mathbf{x}'$, guaranteeing dense output coverage when $T$ is invertible. Interpolation is then required because the recovered source coordinate is usually fractional.


## 16. Translation

Translation moves every point by the same displacement:

```text
x' = x + tx
y' = y + ty
```

Important consequences:

- output pixels can leave the image canvas;
- newly uncovered regions need a fill value;
- image dimensions do not have to change.


## 17. Rotation

Rotation changes orientation.

The standard rotation matrix acts around the origin. For an image, rotating around the center is usually more intuitive.

Because the image $y$-axis points downward, the visual direction associated with the mathematical sign of $\theta$ can appear reversed compared with a standard Cartesian graph.


## 18. Scaling and Resizing

Scaling changes the spatial size of image content.

Uniform scaling uses:

```text
sx = sy
```

and preserves shape proportions.

Non-uniform scaling uses:

```text
sx ≠ sy
```

and changes the aspect ratio of the content.


## 19. Interpolation Comparison

A geometric transform often asks for the input intensity at a coordinate such as:

```text
x = 104.37
y = 83.62
```

But digital images only store samples at integer coordinates.

Interpolation estimates a value between known pixels.

### Nearest neighbor

Uses the closest pixel.

- fastest;
- preserves discrete labels;
- can look blocky.

### Bilinear

Uses the four nearest spatial samples.

- smoother;
- common for ordinary image resizing and warping.

### Bicubic

Uses a larger neighborhood and a cubic model.

- often smoother/sharper;
- more computationally expensive;
- can introduce overshoot before clipping.

### Deeper understanding

For a fractional source coordinate $(x,y)$, nearest-neighbor selects the closest sample. Bilinear interpolation combines the four surrounding samples with separable linear weights. If $x=i+\alpha$ and $y=j+\beta$,

$$
I(x,y)\approx
(1-\alpha)(1-\beta)I_{i,j}
+\alpha(1-\beta)I_{i+1,j}
+(1-\alpha)\beta I_{i,j+1}
+\alpha\beta I_{i+1,j+1}.
$$

Bicubic interpolation uses a $4\times4$ neighborhood and cubic kernels, producing smoother derivatives at greater computational cost. Interpolation changes sampled intensity values even when the geometric transform is exact.


## 20. Reflection / Flipping

A horizontal reflection changes left-right orientation.

The pure reflection matrix acts around the origin, so it must be translated appropriately to remain inside the image canvas.

For width $W$:

```text
x' = (W - 1) - x
```


## 21. Shear

Shearing slants an image.

Horizontal shear:

```text
x' = x + kx*y
y' = y
```

Vertical shear:

```text
x' = x
y' = ky*x + y
```

Shear preserves straight lines but changes angles and shape.


## 22. Composition of Transformations

Several geometric transformations can be combined into one matrix.

If:

$$
\mathbf{p}'=
\mathbf{H}_2\mathbf{H}_1\mathbf{p}
$$

then $\mathbf{H}_1$ acts **first** and $\mathbf{H}_2$ acts **second**.

Matrix multiplication is not commutative:

$$
\mathbf{H}_2\mathbf{H}_1
\neq
\mathbf{H}_1\mathbf{H}_2
$$

Therefore, transformation order matters.

### Deeper understanding

With column-vector convention, a sequence “apply $A$, then $B$” is

$$
\mathbf{x}'=BA\mathbf{x}.
$$

Matrix multiplication is generally non-commutative:

$$
BA\neq AB.
$$

This is why transformation order is part of the model rather than an implementation detail. A composite matrix should be built only after the intended operation order is stated explicitly.


## 23. General Affine Transformation

An affine transformation has the form:

$$
\begin{bmatrix}
x'\\
y'\\
1
\end{bmatrix}
=
\begin{bmatrix}
a&b&t_x\\
c&d&t_y\\
0&0&1
\end{bmatrix}
\begin{bmatrix}
x\\
y\\
1
\end{bmatrix}
$$

Affine transformations can combine:

- translation;
- rotation;
- scaling;
- reflection;
- shear.

A key geometric property is that **parallel lines remain parallel**.

Perspective effects are not represented by a general affine transform; they require a projective homography.

### Deeper understanding

An affine mapping has six independent parameters:

$$
x'=a_{11}x+a_{12}y+t_x,\qquad
y'=a_{21}x+a_{22}y+t_y.
$$

Three non-collinear point correspondences provide six scalar equations and are the algebraic minimum for determining a general affine transform. Nearly collinear control points lead to poor conditioning, so practical estimation benefits from well-spread points and redundancy.


## 24. Resizing to a New Array Shape

So far, the warp examples kept a fixed canvas.

In real workflows, resizing often means creating an output array with a different shape.

Pillow provides explicit resampling methods for this purpose.


## 25. Validation Checks

Transformations should be validated both mathematically and visually.

Useful checks include:

- output dtype;
- output range;
- output shape;
- identity behavior;
- known point transformation;
- matrix invertibility;
- reflection consistency;
- histogram-equalization mapping monotonicity.

### Deeper understanding

Useful invariants depend on the transformation. Identity should reproduce the input; pure translation should preserve local shape; rigid rotation should preserve Euclidean distances before resampling; reflection should reverse orientation; affine transforms should preserve collinearity and parallelism.

Numerical validation should distinguish **geometric correctness** from **resampling error**. A coordinate transform can be exact even when interpolation changes the pixel values.


## Technical Synthesis

The transformation framework separates radiometric mappings from geometric mappings:

$$
\boxed{
\text{image}
\rightarrow
\begin{cases}
\text{intensity mapping}\\
\text{geometric mapping}
\end{cases}
\rightarrow
\text{sampling/interpolation}
\rightarrow
\text{transformed image}
\rightarrow
\text{validation}
}
$$

Intensity transforms operate on pixel values, whereas geometric transforms alter spatial coordinates and therefore require a sampling rule. Homogeneous coordinates, inverse mapping, interpolation, transformation composition, and boundary handling are the principal elements required for numerically stable and geometrically interpretable results.

## Scope and Limitations

### Included

Intensity transformations, histogram equalization, translation, rotation, scaling, reflection, shear, composition, interpolation, and affine warping.

### Not included

Spatial convolution, Fourier-domain filtering, segmentation, and projective homography estimation.


## References

1. **R. C. Gonzalez and R. E. Woods**, *Digital Image Processing* — reference for intensity transformations, histogram processing, interpolation, and geometric image transformations. [Companion site](https://www.imageprocessingplace.com/)
2. **OpenCV Documentation**, “Geometric Transformations of Images” — aligned practical reference for scaling, translation, rotation, affine mappings, perspective mappings, and interpolation choices. [OpenCV tutorial](https://docs.opencv.org/4.x/da/d6e/tutorial_py_geometric_transformations.html)
3. **SciPy Documentation**, `scipy.ndimage.affine_transform` — implementation reference for inverse-coordinate affine resampling and interpolation in multidimensional arrays. [SciPy API](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.affine_transform.html)
4. **Pillow Documentation**, “Concepts” — reference for image modes, coordinates, sizes, and raster representation used by the transformation workflow. [Pillow concepts](https://pillow.readthedocs.io/en/stable/handbook/concepts.html)
5. **Pillow Documentation**, “Image class / transforms” — practical reference for resize and image transformation behavior in Pillow-based workflows. [Pillow Image API](https://pillow.readthedocs.io/en/stable/reference/Image.html)
